# Figure 3 | Cortical AD-up hdWGCNA modules

Publication-ready analysis notebook for the manuscript figure. Exploratory checks, scratch outputs, and analyses unrelated to the figure panels have been removed. Paths are repository-relative; set `NVU_PROJECT_ROOT` when running from another location.

Analysis scope:
- Cortical AD-up DEG gene set preparation for L2/3 and L4-6 NVUs.
- hdWGCNA module construction, module-trait association, TOM heatmaps, and GO enrichment outputs.

In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath("..", mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure3")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
suppressMessages({library(Seurat)
library(tidyverse)
library(cowplot)
library(patchwork)
library(ggplot2)
#library(ggthemes)
library(plyr)
library(WGCNA)
library(hdWGCNA)
library(harmony)
library(corrplot)
library(igraph)
library(UCell)
theme_set(theme_cowplot())})
set.seed(123)
enableWGCNAThreads(nThreads = 16)

best_color <- c("#fb8071", "#99cc31", "#8cd2c7", "#208b23", "#199a73","#cbeac4",#EX
                "#32a02d", "#a4cee1", "#02bdfe","#08529d", "#108b96", "#377EB8",
                "#fcb462", "#fd7c01", "#fdbc66","#da5e05",  "#fd786f",#IN
                "#CAB2D6", "#6A3D9A",  "#BC80BD","#9571d9",
                "#FBB4AE", "#B3CDE3", "#CCEBC5", "#DECBE4", "#FED9A6","#FFFFCC", "#E5D8BD",
                "#FDDAEC", "#F2F2F2", "#B3E2CD", "#FDCDAC","#CBD5E8", "#F4CAE4", "#E6F5C9", "#FFF2AE",
                "#F1E2CC", "#CCCCCC","#E41A1C",  "#984EA3", "#FFFF33", "#A65628", "#F781BF",
                "#999999", "#FFED6F", "#66C2A5", "#FC8D62", "#8DA0CB", "#E78AC3","#A6D854", "#FFD92F",
                "#E5C494", "#B3B3B3", "#8DD3C7", "#FFFFB3","#BEBADA", "#FB8072", "#80B1D3", "#FDB462",
                "#B3DE69", "#FCCDE5","#D9D9D9","#666666")
genelist = read.csv('../results/Figure3/Venn_union_genelist_L23_L456_Up.csv')
length(genelist$gene)

In [ ]:
st_obj = readRDS('../results/05_nvu_analysis_results/Cortex/200/Cortex_merged-subtype_0330.rds')

In [ ]:
st_obj1 = subset(st_obj,area_m%in%c('L23','L456'))
st_obj1 = subset(st_obj1,celltype_unit%in%c("Astro","Endo","Micro","Neuron","Oligo","Pericyte"))

In [ ]:
remove_samples <- c("GSM8330064_B02008C6", "GSM8330062_C02248B5")
st_obj1 <- subset(st_obj1, sample_id %in% remove_samples == FALSE)

In [ ]:
seurat_obj <- SetupForWGCNA( st_obj1, gene_select = "fraction", # the gene selection approach 
                            fraction = 0.001, 
                            features = genelist$gene,# fraction of cells that a gene needs to be expressed in order to be included 
                            wgcna_name = "NVU" # the name of the hdWGCNA experiment 
                           )
print("**************一共考虑的基因个数是#####################")
print(length(seurat_obj@misc$NVU$wgcna_genes))

In [ ]:
seurat_obj$Group = paste0(seurat_obj$celltype_unit,'_',seurat_obj$group)

In [ ]:
seurat_obj <- NormalizeData(object = seurat_obj)
seurat_obj <- FindVariableFeatures(object = seurat_obj, nfeatures = 3000)
seurat_obj <- ScaleData(object = seurat_obj, features = genelist$gene)
seurat_obj <- SCTransform(
  seurat_obj, 
  variable.features.n = NULL,
  variable.features.rv.th = NULL,
  residual.features = genelist$gene,
  verbose = FALSE
)

cat("SCT基因数:", nrow(seurat_obj[["SCT"]]), "\n")
cat("与genelist重叠:", sum(rownames(seurat_obj[["SCT"]]) %in% genelist$gene), "\n")
seurat_obj <- RunPCA(seurat_obj, assay = "SCT", verbose = FALSE,features = genelist$gene)
seurat_obj <- FindNeighbors(seurat_obj, reduction = "pca", dims = 1:30)

In [ ]:
# construct metacells  in each group
seurat_obj <- MetacellsByGroups(
  seurat_obj = seurat_obj,
  group.by = c("group",'celltype_unit'),
  k = 25, ident.group = 'group')#, slot = "counts", max_shared = 10, min_cells = 30)

In [ ]:
seurat_obj <- SetDatExpr(
  seurat_obj,#seurat_obj
  group_name = c("Astro","Endo","Micro","Neuron","Oligo","Pericyte"), # the name of the group of interest in the group.by column
  group.by='celltype_unit', # the metadata column containing the cell type info. This same column should have also been used in MetacellsByGroups
  assay = 'SCT', # using RNA assay
  use_metacells = TRUE#, # use the metacells (TRUE) or the full expression matrix (FALSE)
  # slot = 'data' # using normalized data
)

In [ ]:
# Set the soft threshold
# Test different soft powers:
seurat_obj <- TestSoftPowers(
  seurat_obj,
  #networkType = 'signed', # you can also use "unsigned" or "signed hybrid"
  setDatExpr = FALSE
)
# plot the results:
plot_list <- PlotSoftPowers(seurat_obj,point_size = 5,text_size = 3)
# assemble with patchwork

p = wrap_plots(plot_list, ncol=2)
print(p)

power_table <- GetPowerTable(seurat_obj)
power_table = as.data.frame(power_table)
#write.table(power_table,paste0(output,"/power_table.csv"),row.names = FALSE,sep = ",",quote = FALSE)
saveRDS(seurat_obj,'../results/06_Wgcna/01_WGcna_cellbin_Cortex_L23-L456_NVU.rds')

In [ ]:
seurat_obj = readRDS('../results/06_Wgcna/01_WGcna_cellbin_Cortex_L23-L456_NVU.rds')

In [ ]:
seurat_obj <- ConstructNetwork(
  seurat_obj, soft_power=7,  # Select the soft threshold
  setDatExpr=FALSE,
 #tom_name = "new_TOM_name",
  #tom_name = class, # name of the topoligical overlap matrix written to disk
  corType = "pearson",
  networkType = "signed",
  TOMType = "signed 2",
  detectCutHeight = 0.9999,
  minModuleSize = 15,
  mergeCutHeight = 0.35,
  overwrite_tom = TRUE
)

PlotDendrogram(seurat_obj, main = 'WGCNA Dendrogram')

In [ ]:
modules = seurat_obj@misc$NVU$wgcna_modules
net <- GetNetworkData(seurat_obj)
modules$module <- factor(modules$module)
modules[modules$color=='grey',]$color = '#eaebea'
modules[modules$color=='blue',]$color = '#6e9bc5'
modules[modules$color=='yellow',]$color = '#ffee6f'
modules[modules$color=='turquoise',]$color = '#40E0D0'
modules[modules$color=='brown',]$color = '#A52A2A'
modules[modules$color=='green',]$color = '#199a73'

In [ ]:
options(repr.plot.width=12, repr.plot.height=4)
pdf('../results/06_Wgcna/Cortex_up_Dendrogram_allgene.pdf', width = 12, height = 4)

WGCNA::plotDendroAndColors(
    net$dendrograms[[1]],
    as.character(modules$color),
    groupLabels='Module colors',
    dendroLabels = FALSE,
    hang = 0.03,
    addGuide = TRUE,
    guideHang = 0.05,
    # main = main,

  )
dev.off()

In [ ]:
Gene2Module <- seurat_obj@misc$NVU$wgcna_modules

In [ ]:
# need to run ScaleData first or else harmony throws an error:
seurat_obj$subcelltype <- as.character(seurat_obj$subcelltype)

seurat_obj$subcelltype[seurat_obj$subcelltype %in% 
  c("Micro_0", "Micro_1", "Micro_2", "Micro_3", "Micro_4")] <- "Micro"

seurat_obj$subcelltype[seurat_obj$subcelltype %in% 
  c("Astro_0", "Astro_1", "Astro_2", "Astro_3", "Astro_4")] <- "Astro"

seurat_obj$subcelltype <- factor(seurat_obj$subcelltype)


seurat_obj <- ScaleData(seurat_obj)
# compute all MEs in the full single-cell dataset
seurat_obj <- ModuleEigengenes(
  seurat_obj,scale.model.use = "linear"           ## group.by.vars="Sample"ww
)

# module eigengenes:
MEs <- GetMEs(seurat_obj, harmonized=FALSE)
hMEs <- GetMEs(seurat_obj)
# write.table(MEs,paste0(output,"/MEs.csv"),row.names = FALSE,sep = ",",quote = FALSE)
# saveRDS(seurat_obj, "bin_Sub_CA1.hdWGCNA_object_total.rds")
seurat_obj@meta.data[1:2,]
DMEs_all <- FindAllDMEs(
  seurat_obj,
  group.by = 'subcelltype'
)

In [ ]:
# get the modules table:
modules <- GetModules(seurat_obj)
mods <- levels(modules$module); mods <- mods[mods != 'grey']

library(tidyr)
plot_df = DMEs_all
plot_df1 = plot_df
plot_df1$group_module = row.names(plot_df1)

logfc_df = plot_df1[,c('avg_log2FC','group','module')]
pvalue_df = plot_df1[,c('p_val_adj','group','module')]

#logfc_df
logfc_wide<-tidyr::spread(logfc_df,group,avg_log2FC)
row.names(logfc_wide) = logfc_wide$module
logfc_wide = logfc_wide[,-1]
logfc_wide = data.frame(t(logfc_wide),check.names=F)
# logfc_wide[1:2,]

#pvalue_df
pvalue_wide<-tidyr::spread(pvalue_df,group,p_val_adj)
row.names(pvalue_wide) = pvalue_wide$module
pvalue_wide = pvalue_wide[,-1]
pvalue_wide = data.frame(t(pvalue_wide),check.names=F)

# logfc_wide = logfc_wide[rev(c('Con_CA1','Con_Sub','AD_CA1','AD_Sub')),]

# pvalue_wide = pvalue_wide[,colnames(logfc_wide) ]
# pvalue_wide = pvalue_wide[row.names(logfc_wide), ]

# Determine significance
pmat = pvalue_wide


if(!is.null(pmat) ){
    ssmt <- pmat < 0.001
    pmat[ssmt] <- '***'
    smt <- pmat < 0.01 & pmat >= 0.001
    pmat[smt] <- '**'
    tmat <- pmat < 0.05 & pmat >= 0.01
    pmat[tmat] <- '*'
    pmat[!ssmt & !smt & !tmat]<- ''
}else{
    pmat <- F
}

pmat = as.matrix(pmat)
# pmat

library(ComplexHeatmap)


cell_order <- c("EX", "IN", "Astro", "Micro", "Oligo", "Endo", "Pericyte")
cell_order <- cell_order[cell_order %in% rownames(logfc_wide)]

logfc_wide  <- logfc_wide[cell_order, , drop = FALSE]
pvalue_wide <- pvalue_wide[cell_order, colnames(logfc_wide), drop = FALSE]

peak_group <- apply(logfc_wide, 2, function(x) {
  rownames(logfc_wide)[which.max(x)]
})

peak_value <- apply(logfc_wide, 2, max, na.rm = TRUE)

module_order <- names(
  sort(
    setNames(match(peak_group, cell_order), colnames(logfc_wide)),
    na.last = TRUE
  )
)

module_order <- module_order[
  order(
    match(peak_group[module_order], cell_order),
    -peak_value[module_order]
  )
]

logfc_wide  <- logfc_wide[, module_order, drop = FALSE]
pvalue_wide <- pvalue_wide[, module_order, drop = FALSE]

pmat <- matrix("", nrow = nrow(pvalue_wide), ncol = ncol(pvalue_wide))
rownames(pmat) <- rownames(pvalue_wide)
colnames(pmat) <- colnames(pvalue_wide)

pmat[pvalue_wide < 0.05]  <- "*"
pmat[pvalue_wide < 0.01]  <- "**"
pmat[pvalue_wide < 0.001] <- "***"

col_fun = circlize::colorRamp2(c(-6, 0,2),  c("#588dd5", "#fbfbfb", "#f5994e") ) #c("#3C5088","white","#A6443D"))
p = Heatmap(logfc_wide ,cluster_rows = F,cluster_columns = F,#row_split = row.type,#column_order = order(colnames(mat_scaled)),
          # left_annotation = left_anno,
            cell_fun = function(j, i, x, y, width, height, fill) {
                # Get star labels for the corresponding p-value matrix positions
                star_symbol = pmat[i, j]
                grid.text(star_symbol, x, y, gp = gpar(fontsize = 10))
            },
            width = ncol(logfc_wide)*unit(10, "mm"), height = nrow(logfc_wide)*unit(10, "mm"),
            col = col_fun,column_names_rot = 75 )

p

pdf('../results/06_Wgcna/02.Cortex_up_group_area.module.cor.pdf',width = 6,height = 6)
p
dev.off()

In [ ]:
# Check whether MEs have been added


# Ensure row names are aligned

MEs_aligned <- MEs[colnames(seurat_obj), , drop = FALSE]

meta_clean <- seurat_obj@meta.data[, !colnames(seurat_obj@meta.data) %in% colnames(MEs)]
seurat_obj@meta.data <- cbind(meta_clean, MEs_aligned)

# Confirm module column names
mods <- colnames(MEs_aligned)
mods <- mods[mods != "grey"]


options(repr.plot.width=12, repr.plot.height=8)
# X-axis order (cell types)
celltype_order <- c("EX", "IN", "Astro", "Micro", "Oligo", "Endo", "Pericyte")


module_order <- c(
  "brown",     
  "blue",      
  "yellow", 
      "green", 
  "turquoise" 

)


seurat_obj$Group <- paste0(seurat_obj$subcelltype, "_", seurat_obj$group)

cells_keep <- rownames(seurat_obj@meta.data)[
  seurat_obj$subcelltype %in% celltype_order
]
seurat_sub <- subset(seurat_obj, cells = cells_keep)


seurat_obj <- subset(seurat_obj, 
  sample_id %in% c("GSM8330064_B02008C6", "GSM8330062_C02248B5") == FALSE)

group_order <- paste0(
  rep(celltype_order, each = 2),
  "_",
  rep(c("Control", "AD"), times = length(celltype_order))
)
seurat_sub$Group <- factor(seurat_sub$Group, levels = group_order)


# Plotting
p <- DotPlot(
  seurat_sub,
  features = rev(module_order),
  group.by = 'Group',
  dot.min  = 0.1
) +
  coord_flip() +
  scale_color_gradient2(
    high = '#f5994e', mid = '#fbfbfb', low = '#588dd5',
    name = "Module\nEigengene"
  ) +
  theme_classic() +
  theme(
    axis.text.y = element_text(size = 11, face = "bold"),
    axis.text.x = element_text(size = 11, face = "bold",
                               angle = 45, hjust = 1, vjust = 1),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 13),
    panel.grid.major = element_line(color = "grey92", linewidth = 0.3)
  ) +
  xlab('') + ylab('') +
  labs(title = "Module Eigengene by Cell Type")

print(p)
ggsave('../results/06_Wgcna/Cortex_up_dotplot_module_ordered.pdf',
       height = 5, width = 9)

In [ ]:
TOM <- GetTOM(seurat_obj)
saveRDS(TOM,'../results/06_Wgcna/TOM/Cortex_NVU_tom.rds')

In [ ]:
library(pheatmap)
tom_matrix = readRDS('../results/06_Wgcna/TOM/Cortex_NVU_tom.rds')
gene_info = read.csv('../results/06_Wgcna/Cortex_up_NVU.Module.csv')
gene_info = gene_info[!gene_info$module=='grey',]

# Prepare annotation data
annotation_df <- data.frame(
  Module = gene_info[order(gene_info$module), "module"],
  row.names = gene_info[order(gene_info$module), "gene_name"]
)


gene_info_sorted <- gene_info[order(gene_info$module), ]
gene_order <- gene_info[order(gene_info$module), "gene_name"]
tom_matrix <- tom_matrix[gene_order, gene_order]

module_counts <- rle(gene_info_sorted$module)$lengths
gaps_positions <- cumsum(module_counts)[-length(module_counts)]


library(pheatmap)
gene_info$module = factor(gene_info$module,levels = c('pink','yellow','turquoise','green','blue','magenta','brown','black','red'))
annotation_df <- data.frame(
  Module = gene_info[order(gene_info$module), "module"],
  row.names = gene_info[order(gene_info$module), "gene_name"]
)

module_colors <- list(
  Module = c(
     pink = "#e15987",
    black = "#fd7c01",
    brown = "brown",
    blue = "#6e9bc5",
    magenta = "#984EA3",
    red = "#e94829",
    green = "#199a73",
    yellow = "#ffee6f",
    turquoise = "turquoise"

  )
)
gene_info_sorted <- gene_info[order(gene_info$module), ]
gene_order <- gene_info[order(gene_info$module), "gene_name"]
tom_matrix <- tom_matrix[gene_order, gene_order]

clustered_gene_order <- c()

for (module_name in unique(gene_info_sorted$module)) {
  module_genes <- gene_info_sorted$gene_name[gene_info_sorted$module == module_name]
  module_matrix <- tom_matrix[module_genes, module_genes]

  diag(module_matrix) <- NA

  # Clustering
  hc <- hclust(as.dist(1 - module_matrix), method = "ward.D2")

  module_clustered_genes <- module_genes[hc$order]
  clustered_gene_order <- c(clustered_gene_order, module_clustered_genes)
}

tom_filtered <- tom_matrix[clustered_gene_order, clustered_gene_order]
diag(tom_filtered) <- NA
tom_filtered = scale(tom_filtered, center = TRUE, scale = TRUE)  # Normalize

annotation_df_reordered <- annotation_df[clustered_gene_order, , drop = FALSE]  # Re-Sortannotation
annotation_df_reordered$Module = factor(annotation_df_reordered$Module ,levels = c('purple','pink','yellow','turquoise','green','blue','magenta','brown','black','red'))

module_reordered <- as.character(annotation_df_reordered$Module)

module_counts_reordered <- rle(module_reordered)$lengths
gaps_positions_reordered <- cumsum(module_counts_reordered)[-length(module_counts_reordered)]

In [ ]:
# Properly defined color palettes (only one should be active)
mycolors <- rev(c("#57121d", "#57121d","#57121d", "#57121d", "#57121d", "#57121d", "#d56e5e", "#eaebea", "#5390b5", "#1f294e", "#1f294e", "#1f294e", "#1f294e", "#1f294e", "#1f294e"))
options(repr.plot.width=7, repr.plot.height=6)

p <- pheatmap(
  tom_filtered,
  scale = 'row',
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_row = annotation_df_reordered,
  annotation_col = annotation_df_reordered,
  annotation_colors = module_colors,
  show_rownames = FALSE,
  show_colnames = FALSE,
  gaps_row = gaps_positions_reordered,
  gaps_col = gaps_positions_reordered,
  color = colorRampPalette(mycolors)(100),
  main = "Gene Correlation Matrix (Clustered within Modules)",
  border_color = NA
)
p

In [ ]:
# Define the color sequence and ensure parentheses are closed
palette_colors <- rev(c(
  "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152",
  "#C72582",  "#FAEAF2",
   "#EDF5E1","#529624", 
  "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419"
))
library(pheatmap)
# Create the color palette with colorRampPalette
color_palette <- colorRampPalette(palette_colors)(100)

# Generate heatmap
p = pheatmap(
  tom_filtered,
  scale = 'row',
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_row = annotation_df_reordered,
  annotation_col = annotation_df_reordered,
  annotation_colors = module_colors,
  show_rownames = FALSE,
  show_colnames = FALSE,
  gaps_row = gaps_positions_reordered,
  gaps_col = gaps_positions_reordered,
  # breaks = seq(0, quantile(tom_matrix, 0.1), length.out = 100),
  color = color_palette,
  main = "Gene Correlation Matrix (Clustered within Modules)",
  border_color = NA
)
p

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
library(dplyr)
library(ggplot2)

# 1. Preparedata
cat("模块分布:\n")

modules <- unique(gene_info$module)
cat("\n共", length(modules), "个模块\n")

all_genes_entrez <- bitr(
  unique(gene_info$gene_name),
  fromType = "SYMBOL",
  toType   = "ENTREZID",
  OrgDb    = org.Hs.eg.db
)
cat("\n基因转换成功:", nrow(all_genes_entrez), "/", length(unique(gene_info$gene_name)), "\n")

gene_info_entrez <- gene_info %>%
  dplyr::left_join(all_genes_entrez,
                   by = c("gene_name" = "SYMBOL")) %>%
  dplyr::filter(!is.na(ENTREZID))

go_results <- list()

for(mod in modules) {
  cat("处理模块:", mod, "...")
  
  mod_genes <- gene_info_entrez$ENTREZID[gene_info_entrez$module == mod]
  cat("基因数:", length(mod_genes), "\n")
  
  if(length(mod_genes) < 5) {
    cat("  基因数太少，跳过\n")
    next
  }
  
  tryCatch({
    ego <- enrichGO(
      gene          = mod_genes,
      OrgDb         = org.Hs.eg.db,
      ont           = "BP",
      pAdjustMethod = "BH",
      pvalueCutoff  = 0.05,
      qvalueCutoff  = 0.2,
      readable      = TRUE
    )
    
    if(nrow(ego@result) > 0) {
      go_results[[mod]] <- ego
      cat("  显著GO terms:", sum(ego@result$p.adjust < 0.05), "\n")
    } else {
      cat("  无显著结果\n")
    }
  }, error = function(e) {
    cat("  错误:", e$message, "\n")
  })
}

cat("\n有结果的模块数:", length(go_results), "\n")

go_combined <- dplyr::bind_rows(
  lapply(names(go_results), function(mod) {
    go_results[[mod]]@result %>%
      dplyr::filter(p.adjust < 0.05) %>%
      dplyr::mutate(module = mod) %>%
      dplyr::slice_head(n = 10)
  })
)

cat("\n合并后总GO terms数:", nrow(go_combined), "\n")

top5_per_module <- go_combined %>%
  dplyr::group_by(module) %>%
  dplyr::arrange(p.adjust) %>%
  dplyr::slice_head(n = 5) %>%
  dplyr::ungroup() %>%
  dplyr::mutate(
    Description = stringr::str_wrap(Description, width = 40),
    log_padj    = -log10(p.adjust),
    GeneRatio_num = sapply(GeneRatio, function(x) {
      parts <- strsplit(x, "/")[[1]]
      as.numeric(parts[1]) / as.numeric(parts[2])
    })
  )

# Module color mapping
module_colors <- setNames(
  unique(gene_info$color[gene_info$module %in% names(go_results)]),
  unique(gene_info$module[gene_info$module %in% names(go_results)])
)

p_dot <- ggplot(
  top5_per_module,
  aes(x = module, y = reorder(Description, log_padj))
) +
  geom_point(
    aes(size  = Count,
        color = log_padj)
  ) +
  scale_color_gradient(
    low  = "#FEE0D2",
    high = "#E64B35",
    name = "-log10\n(p.adjust)"
  ) +
  scale_size_continuous(
    name  = "Gene Count",
    range = c(2, 8)
  ) +
  facet_wrap(~ module, scales = "free_y", ncol = 3) +
  theme_classic() +
  theme(
    axis.text.x      = element_text(size = 10, face = "bold"),
    axis.text.y      = element_text(size = 8),
    strip.text       = element_text(size = 11, face = "bold"),
    strip.background = element_rect(fill = "grey90", color = NA),
    plot.title       = element_text(hjust = 0.5, face = "bold", size = 13),
    panel.spacing    = unit(1, "lines"),
    legend.position  = "right"
  ) +
  labs(
    title = "GO-BP Enrichment by WGCNA Module (Top 5)",
    x     = "", y = ""
  )

print(p_dot)
ggsave('../results/06_Wgcna/Cortex_GO_BP_dotplot_by_module.pdf',
       p_dot,
       width  = 5 * min(length(go_results), 3),
       height = max(length(go_results) / 3 * 4, 10),
       limitsize = FALSE)

In [ ]:
magenta_genes = gene_info[gene_info$module=='magenta',]$gene_name

magenta_entrez <- bitr(
  magenta_genes,
  fromType = "SYMBOL",
  toType   = "ENTREZID",
  OrgDb    = org.Hs.eg.db
)$ENTREZID

ego_relaxed <- enrichGO(
  gene          = magenta_entrez,
  OrgDb         = org.Hs.eg.db,
  ont           = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff  = 1,  # no filtering
  qvalueCutoff  = 1,  # no filtering
  readable      = TRUE
)

vascular_keywords <- c(
  "vascular", "vessel", "angiogen", "endothel", "pericyte",
  "smooth muscle", "blood", "circulat",
  "glucose", "transport", "barrier",
  "contraction", "actomyosin"
)

result_df <- ego_relaxed@result %>%
  dplyr::filter(
    grepl(paste(vascular_keywords, collapse = "|"),
          Description, ignore.case = TRUE)
  ) %>%
  dplyr::arrange(pvalue) %>%
  dplyr::select(Description, GeneRatio, pvalue, p.adjust, geneID, Count)

cat("\n血管相关GO terms（不限显著性）:\n")

vascular_markers <- c(
  "RGS5", "PDGFRB", "ACTA2", "MYH11",  # Pericytes/SMC
  "CLDN5", "PECAM1", "CDH5", "VWF",  # Endothelial
  "SLC2A1", "ABCB1", "OCLN",                  # BBB
  "MYL9", "MYL12A", "TAGLN2", "CNN1"
)

overlap_vascular <- intersect(magenta_genes, vascular_markers)
cat("\nMagenta与血管标志物重叠:\n")
cat(paste(overlap_vascular, collapse = ", "), "\n")
cat("覆盖率:", length(overlap_vascular), "/", length(vascular_markers), "\n")

plot_df <- ego_relaxed@result %>%
  dplyr::arrange(pvalue) %>%
  dplyr::slice_head(n = 20) %>%
  dplyr::mutate(
    log_p     = -log10(pvalue),
    sig_label = ifelse(p.adjust < 0.05, "Significant", "Trend"),
    term_wrap = stringr::str_wrap(Description, 35)
  )

p_magenta <- ggplot(
  plot_df,
  aes(x = log_p,
      y = reorder(term_wrap, log_p),
      fill = sig_label)
) +
  geom_col(width = 0.7, alpha = 0.85) +
  geom_vline(
    xintercept = -log10(0.05),
    linetype = "dashed", color = "grey40", linewidth = 0.8
  ) +
  scale_fill_manual(
    values = c("Significant" = "#E64B35", "Trend" = "#F39B7F"),
    name   = ""
  ) +
  theme_classic() +
  theme(
    axis.text.y     = element_text(size = 9),
    axis.text.x     = element_text(size = 10),
    plot.title      = element_text(hjust = 0.5, face = "bold", size = 12),
    plot.subtitle   = element_text(hjust = 0.5, size = 9, color = "grey40"),
    legend.position = "top"
  ) +
  labs(
    title    = "Magenta Module: GO-BP Enrichment",
    subtitle = "Dashed line: p=0.05 | Dark=significant, Light=trend",
    x        = "-log10(pvalue)", y = ""
  )

p_magenta

In [ ]:
write.csv(result_df,'../results/06_Wgcna/GO_BP_magenta.csv')

In [ ]:
write.csv(go_combined,
          '../results/06_Wgcna/GO_BP_all_modules.csv',
          row.names = FALSE)

## Hub-gene network panels

In [ ]:
library(Matrix)
library(igraph)
library(scales)

module_table <- read.csv('../results/06_Wgcna/Cortex_up_NVU.Module.csv')
modules_df <- module_table %>%
  dplyr::filter(module != "grey") %>%
  dplyr::select(gene_name, module, color) %>%
  dplyr::distinct()

seurat_obj@misc$wgcna_modules <- modules_df

sct_scale_genes <- rownames(seurat_obj@assays$SCT@scale.data)
genes_use <- modules_df$gene_name[modules_df$gene_name %in% rownames(seurat_obj@assays$SCT@data)]
genes_in_sct <- genes_use[genes_use %in% sct_scale_genes]
genes_in_data <- genes_use[genes_use %in% rownames(seurat_obj@assays$SCT@data)]

if (length(genes_in_sct) > 100) {
  expr_for_network <- t(seurat_obj@assays$SCT@scale.data[genes_in_sct, ])
} else {
  expr_for_network <- t(as.matrix(seurat_obj@assays$SCT@data[genes_in_data, ]))
}

cells_common <- intersect(rownames(expr_for_network), rownames(MEs))
expr_net_aligned <- expr_for_network[cells_common, , drop = FALSE]
MEs_aligned <- MEs[cells_common, , drop = FALSE]

mod_colors <- setdiff(unique(modules_df$module), "grey")
kME_list <- list()

for (mod in mod_colors) {
  mod_genes_cur <- modules_df$gene_name[modules_df$module == mod]
  mod_genes_cur <- mod_genes_cur[mod_genes_cur %in% colnames(expr_net_aligned)]
  if (length(mod_genes_cur) == 0 || !mod %in% colnames(MEs_aligned)) next

  expr_mod <- as.matrix(expr_net_aligned[, mod_genes_cur, drop = FALSE])
  me_vec <- MEs_aligned[[mod]]
  kme_vals <- cor(expr_mod, me_vec, use = "pairwise.complete.obs")[, 1]

  kME_list[[mod]] <- data.frame(
    gene_name = names(kme_vals),
    kME = kme_vals,
    module = mod,
    stringsAsFactors = FALSE
  )
}

modules_with_kme <- modules_df %>%
  dplyr::left_join(
    dplyr::bind_rows(kME_list) %>%
      tidyr::pivot_wider(names_from = module, values_from = kME, names_prefix = "kME_"),
    by = "gene_name"
  )

network_out_dir <- file.path(RESULTS_DIR, "06_Wgcna", "module_networks_cortex_up")
dir.create(network_out_dir, recursive = TRUE, showWarnings = FALSE)

for (mod in mod_colors) {
  kme_col <- paste0("kME_", mod)
  if (!kme_col %in% colnames(modules_with_kme)) next

  mod_data <- modules_with_kme %>%
    dplyr::filter(module == mod) %>%
    dplyr::arrange(desc(abs(!!sym(kme_col))))

  top_genes <- head(mod_data$gene_name, min(25, nrow(mod_data)))
  top_genes <- top_genes[top_genes %in% colnames(expr_net_aligned)]
  if (length(top_genes) < 3) next

  expr_top <- as.matrix(expr_net_aligned[, top_genes, drop = FALSE])
  cor_mat <- cor(expr_top, use = "pairwise.complete.obs")
  diag(cor_mat) <- 0

  cor_vals <- abs(cor_mat[upper.tri(cor_mat)])
  thresh <- max(quantile(cor_vals, 0.6, na.rm = TRUE), 0.05)
  cor_mat[abs(cor_mat) < thresh] <- 0

  g <- igraph::graph_from_adjacency_matrix(
    cor_mat, mode = "undirected", weighted = TRUE, diag = FALSE
  )

  kme_vals <- mod_data[[kme_col]][match(igraph::V(g)$name, mod_data$gene_name)]
  kme_vals[is.na(kme_vals)] <- 0.1
  node_size <- scales::rescale(abs(kme_vals), to = c(6, 20))

  n_edges <- igraph::ecount(g)
  edge_width <- if (n_edges > 0 && !is.null(igraph::E(g)$weight)) {
    scales::rescale(abs(igraph::E(g)$weight), to = c(0.5, 5))
  } else {
    rep(1, max(n_edges, 1))
  }

  pdf(file.path(network_out_dir, paste0(mod, "_network.pdf")), width = 9, height = 9)
  par(mar = c(1, 1, 3, 1))
  plot(
    g,
    vertex.size = node_size,
    vertex.color = adjustcolor(mod, alpha.f = 0.85),
    vertex.label = igraph::V(g)$name,
    vertex.label.cex = 0.8,
    vertex.label.color = "black",
    vertex.frame.color = "white",
    edge.width = edge_width,
    edge.color = adjustcolor("grey40", alpha.f = 0.5),
    layout = igraph::layout_with_fr(g),
    main = paste0("Module: ", mod, " (", nrow(mod_data), " genes, ", n_edges, " edges)")
  )
  dev.off()
}